In [1]:
%pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 3.0 MB/s  0:00:00

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Prepare and describe the data

In [2]:
import json
with open("datas/train.jsonl", "r") as f:
    data_train = [json.loads(line) for line in f]


with open("datas/test.jsonl", "r") as f:
    data_test = [json.loads(line) for line in f]

with open("datas/devel.jsonl", "r") as f:
    data_devel = [json.loads(line) for line in f]

In [1]:
import pandas as pd
train_df = pd.DataFrame(data_train)
test_df = pd.DataFrame(data_test)

NameError: name 'data_train' is not defined

In [16]:
train_df["intent"]

0           calendar_set
1           calendar_set
2        audio_volume_up
3        iot_hue_lightup
4          weather_query
              ...       
11509     takeaway_query
11510     datetime_query
11511        email_query
11512     play_audiobook
11513       calendar_set
Name: intent, Length: 11514, dtype: str

In [17]:
train_df["intent"].value_counts()

intent
calendar_set      804
play_music        620
weather_query     559
calendar_query    558
general_quirky    546
                 ... 
events              1
convert             1
cleaning            1
settings            1
volume_other        1
Name: count, Length: 91, dtype: int64

In [27]:
df = pd.concat([train_df[["sentence", "intent"]],test_df[["sentence", "intent"]], pd.DataFrame(data_devel)[["sentence", "intent"]]], axis=0, ignore_index=True)

In [28]:
df = df.groupby("intent").filter(lambda x: len(x) >= 2)

In [29]:
df.describe(include="all")

,sentence,intent
count,16510,16510
unique,16486,82
top,do i have any new email,calendar_set
freq,3,1142


In [30]:
df.to_json("datas/df_slurp.jsonl", orient="records", lines=True, force_ascii=False)

In [31]:
df

,sentence,intent
0,event,calendar_set
1,i need an event three days from now scheduled ...,calendar_set
2,turn up,audio_volume_up
3,brighten the lights a little bit,iot_hue_lightup
4,what's the current weather,weather_query
...,...,...
16516,can you give me local news on wayne county she...,news_query
16517,every light of room increase its intensity,iot_hue_lightup
16518,i would like some coffee now,iot_coffee
16519,what is the population of los angeles,qa_factoid


In [48]:
df = (
    df.groupby("intent", group_keys=False).filter(lambda x: len(x) >= 5).groupby("intent", group_keys=False).sample(n=5, random_state=42))

In [49]:
df["intent"].value_counts()

intent
alarm_query          5
alarm_remove         5
alarm_set            5
audio_volume_down    5
audio_volume_mute    5
                    ..
transport_query      5
transport_taxi       5
transport_ticket     5
transport_traffic    5
weather_query        5
Name: count, Length: 75, dtype: int64

In [7]:
df.to_json("datas/df_slurp.jsonl", orient="records", lines=True, force_ascii=False)

NameError: name 'df' is not defined

In [9]:
import pandas as pd

In [24]:
df = pd.read_json("datas/df_train.jsonl", orient="records", lines=True)

In [25]:
df["intent"].value_counts()

intent
alarm_query                 12
alarm_remove                12
alarm_set                   12
audio_volume_down           12
audio_volume_mute           12
audio_volume_up             12
calendar_query              12
calendar_remove             12
calendar_set                12
cooking_query               12
cooking_recipe              12
datetime_convert            12
datetime_query              12
email_addcontact            12
email_query                 12
email_querycontact          12
email_sendemail             12
general_greet               12
general_joke                12
iot_cleaning                12
iot_coffee                  12
iot_hue_lightchange         12
iot_hue_lightoff            12
iot_hue_lighton             12
iot_wemo_off                12
iot_wemo_on                 12
lists_createoradd           12
lists_query                 12
lists_remove                12
music_query                 12
news_query                  12
play_audiobook              12
p

## Pipeline

In [1]:
import pandas as pd
df = pd.read_json("datas/df_train.jsonl", orient="records", lines=True)
import numpy as np

def prepare_model(model):
    example_vectors = model.encode(
        df["sentence"].tolist(), normalize_embeddings=True, show_progress_bar=False)
    action_vectors = np.stack([
        example_vectors[df["intent"].to_numpy() == intent].mean(axis=0)
        for intent in intents
    ])
    action_vectors /= np.linalg.norm(action_vectors, axis=1, keepdims=True)
    return action_vectors

In [2]:
from sklearn.metrics import accuracy_score, f1_score
THRESHOLD = 0.6
import numpy as np
df_test = pd.read_json("datas/df_test.jsonl", orient="records", lines=True)
df_train = pd.read_json("datas/df_train.jsonl", orient="records", lines=True)
intents = sorted(df_train["intent"].unique())

def test_model(model, action_vectors_for_model):
    assert set(df_test["intent"]) == set(intents)
    assert df_test["intent"].value_counts().eq(3).all()

    test_vectors = model.encode(
        df_test["sentence"].tolist(), normalize_embeddings=True, show_progress_bar=True
    )
    similarities = test_vectors @ action_vectors_for_model.T
    best_indices = np.argmax(similarities, axis=1)
    best_scores = np.clip(similarities[np.arange(len(best_indices)), best_indices], 0, 1)
    y_pred = np.asarray(intents, dtype=object)[best_indices]
    y_pred[best_scores < THRESHOLD] = "unknown"
    y_true = df_test["intent"].to_numpy()

    test_accuracy = accuracy_score(y_true, y_pred)
    test_f1_macro = f1_score(y_true, y_pred, labels=intents, average="macro", zero_division=0)
    metrics = {"accuracy": test_accuracy, "f1_macro": test_f1_macro}
    print(f"Test accuracy: {test_accuracy}")
    print(f"Test macro F1: {test_f1_macro}")
    assert all(df_test["intent"].iloc[i:i + 3].nunique() == 1 for i in range(0, len(df_test), 3))
    for offset, language in enumerate(("Russian", "Kazakh", "English")):
        mask = np.arange(len(df_test)) % 3 == offset
        language_accuracy = accuracy_score(y_true[mask], y_pred[mask])
        print(f"{language} accuracy: {language_accuracy:.4f}")
        metrics[f"accuracy_{language.lower()}"] = language_accuracy

    return metrics


In [3]:
def test_pipe(model):
    action_vectors_for_model = prepare_model(model)
    return test_model(model, action_vectors_for_model)

In [4]:
import pandas as pd

df_small = pd.read_json("datas/df_small.jsonl", orient="records", lines=True)
df_small_train = pd.read_json("datas/df_small_train.jsonl", orient="records", lines=True)
df_small_test = pd.read_json("datas/df_small_test.jsonl", orient="records", lines=True)
print(f"Small dataset: {len(df_small)} examples, "
      f"{len(df_small_train)} train, {len(df_small_test)} test, "
      f"{df_small['intent'].nunique()} intents")


Small dataset: 60 examples, 48 train, 12 test, 4 intents


In [ ]:
def test_pipe_small(model, threshold=0.60):
    import numpy as np
    from sklearn.metrics import accuracy_score, f1_score

    intents_small = sorted(df_small_train["intent"].unique())
    train_vectors = model.encode(df_small_train["sentence"].tolist(), normalize_embeddings=True)
    centers = np.stack([
        train_vectors[df_small_train["intent"].to_numpy() == intent].mean(axis=0)
        for intent in intents_small
    ])
    centers /= np.linalg.norm(centers, axis=1, keepdims=True)

    test_vectors = model.encode(df_small_test["sentence"].tolist(), normalize_embeddings=True)
    scores = test_vectors @ centers.T
    predictions = np.array(intents_small, dtype=object)[scores.argmax(axis=1)]
    predictions[scores.max(axis=1) < threshold] = "unknown"
    answers = df_small_test["intent"].to_numpy()

    print(f"Test accuracy: {accuracy_score(answers, predictions)}")
    print(f"Test macro F1: {f1_score(answers, predictions, labels=intents_small, average='macro', zero_division=0)}")
    for offset, language in enumerate(("Russian", "Kazakh", "English")):
        indices = np.arange(len(answers)) % 3 == offset
        print(f"{language} accuracy: {accuracy_score(answers[indices], predictions[indices])}")


In [6]:
from sentence_transformers import SentenceTransformer

In [7]:
import numpy as np

model_names = [
    "BAAI/bge-m3",
    "intfloat/multilingual-e5-small",
    "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-large",
    "Hiveurban/multilingual-e5-large-pooled",
    "Darmm/darmm-embed-kazakh-v2",
    "intfloat/multilingual-e5-large-instruct",
    "nomic-ai/nomic-embed-text-v2-moe",
    "Qwen/Qwen3-Embedding-0.6B",
]


In [8]:
results = []

for model_name in model_names:
    print(model_name)
    model = SentenceTransformer(model_name)
    metrics = test_pipe(model=model)
    results.append({"model": model_name, **metrics})

results_df = pd.DataFrame(results)
results_df


BAAI/bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Test accuracy: 0.82
Test macro F1: 0.8082857142857143
Russian accuracy: 0.8200
Kazakh accuracy: 0.7800
English accuracy: 0.8600
intfloat/multilingual-e5-small


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Test accuracy: 0.5666666666666667
Test macro F1: 0.6239523809523809
Russian accuracy: 0.6400
Kazakh accuracy: 0.3000
English accuracy: 0.7600


,model,accuracy,f1_macro,accuracy_russian,accuracy_kazakh,accuracy_english
0,BAAI/bge-m3,0.820000,0.808286,0.82,0.78,0.86
1,intfloat/multilingual-e5-small,0.560000,0.542004,0.58,0.48,0.62
2,intfloat/multilingual-e5-base,0.700000,0.666841,0.76,0.64,0.70
3,intfloat/multilingual-e5-large,0.746667,0.738004,0.78,0.66,0.80
4,Hiveurban/multilingual-e5-large-pooled,0.746667,0.738004,0.78,0.66,0.80
5,Darmm/darmm-embed-kazakh-v2,0.506667,0.505766,0.50,0.46,0.56
6,intfloat/multilingual-e5-large-instruct,0.800000,0.781846,0.88,0.68,0.84
7,nomic-ai/nomic-embed-text-v2-moe,0.206667,0.193205,0.22,0.20,0.20
8,Qwen/Qwen3-Embedding-0.6B,0.566667,0.623952,0.64,0.30,0.76


In [9]:
results_df.sort_values(by="f1_macro", ascending=False, inplace=True)

In [10]:
results_df

,model,accuracy,f1_macro,accuracy_russian,accuracy_kazakh,accuracy_english
0,BAAI/bge-m3,0.820000,0.808286,0.82,0.78,0.86
6,intfloat/multilingual-e5-large-instruct,0.800000,0.781846,0.88,0.68,0.84
3,intfloat/multilingual-e5-large,0.746667,0.738004,0.78,0.66,0.80
4,Hiveurban/multilingual-e5-large-pooled,0.746667,0.738004,0.78,0.66,0.80
2,intfloat/multilingual-e5-base,0.700000,0.666841,0.76,0.64,0.70
8,Qwen/Qwen3-Embedding-0.6B,0.566667,0.623952,0.64,0.30,0.76
1,intfloat/multilingual-e5-small,0.560000,0.542004,0.58,0.48,0.62
5,Darmm/darmm-embed-kazakh-v2,0.506667,0.505766,0.50,0.46,0.56
7,nomic-ai/nomic-embed-text-v2-moe,0.206667,0.193205,0.22,0.20,0.20


In [ ]:
model = SentenceTransformer(model_names[0])
test_pipe_small(model=model, threshold=0.40)

In [ ]:
def predict(query, model):
    action_vectors_for_model = prepare_model(model)

    query_vector = model.encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    )[0]
    similarities = action_vectors_for_model @ query_vector

    best = int(np.argmax(similarities))
    confidence = float(np.clip(similarities[best], 0, 1))

    return {
        "endpoint": intents[best] if confidence >= THRESHOLD else "unknown",
        "confidence": round(confidence, 4),
    }

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import time
app = FastAPI()
class QueryRequest(BaseModel):
    query: str
@app.post('/classify')
def classify(request: QueryRequest):
    try:
        start_time = time.time()
        result = predict(request.query)
        end_time = time.time()
        print(f"Query processing time: {end_time - start_time:.2f} seconds")
        return result
    except ValueError as error:
        raise HTTPException(status_code=422, detail=str(error)) from error

from fastapi.testclient import TestClient
with TestClient(app) as client:
    print(client.post('/classify', json={'query': 'four am'}).json())

In [ ]:
async def serve():
    import uvicorn
    server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=8000))
    await server.serve()